# 🏎️ F1 Telemetry Comparison Tool

Compare two drivers' fastest laps using real telemetry data from the FastF1 library.

---

### How to use this notebook:
1. Run **Cell 1** — installs and imports all dependencies
2. Run **Cell 2** — configures your session (year, GP, session type, drivers)
3. Run **Cell 3** — loads and caches the session data from FastF1
4. Run **Cell 4** — extracts and aligns telemetry for both drivers
5. Run **Cell 5** — generates all telemetry plots
6. Run **Cell 6** — prints analytical insights summary

> **Tip:** The first load of a session takes ~30–60 seconds. Subsequent loads use the local cache and are instant.


## Cell 1 — Setup: Install & Import Dependencies

In [1]:
# Uncomment to install dependencies on first run:
# !pip install fastf1 plotly pandas numpy notebook ipywidgets

import warnings
warnings.filterwarnings('ignore')

import fastf1
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from pathlib import Path

# Cache prevents re-downloading session data on subsequent runs
CACHE_DIR = Path('cache')
CACHE_DIR.mkdir(exist_ok=True)
fastf1.Cache.enable_cache(str(CACHE_DIR))

pio.renderers.default = 'jupyterlab'

print(f'FastF1 {fastf1.__version__} ready. Cache: {CACHE_DIR.resolve()}')

✅ All imports successful.
   FastF1 version: 3.8.1
   Cache directory: C:\Users\Lenovo User\Desktop\f1-telemetry\cache


## Cell 2 — Configuration: Choose Your Session & Drivers

Edit the values in this cell to change the comparison. 
Then re-run cells 3 → 6 to refresh all data and charts.

In [25]:
# ── Session ───────────────────────────────────────────────────────────────────
YEAR         = 2026       # 2018 onwards
GRAND_PRIX   = 'China'   # e.g. 'Monaco', 'Monza', 'Silverstone', 'Suzuka'
SESSION_TYPE = 'Q'        # 'Q' = Qualifying | 'R' = Race | 'FP1' 'FP2' 'FP3'

# ── Drivers ───────────────────────────────────────────────────────────────────
DRIVER_1 = 'RUS'          # e.g. 'VER', 'HAM', 'LEC', 'NOR', 'SAI'
DRIVER_2 = 'ANT'

# ── Visual ────────────────────────────────────────────────────────────────────
COLOR_D1    = '#00d2ff'
COLOR_D2    = '#ff6b35'
CHART_THEME = 'plotly_dark'  # 'plotly_dark' or 'plotly_white'

# ── Export ────────────────────────────────────────────────────────────────────
SAVE_HTML  = True
EXPORT_DIR = Path('exports')
EXPORT_DIR.mkdir(exist_ok=True)

print(f'{YEAR} {GRAND_PRIX} GP — {SESSION_TYPE} | {DRIVER_1} vs {DRIVER_2}')

2026 China GP — Q | RUS vs ANT


## Cell 3 — Load Session Data

FastF1 fetches session data from the official F1 data API.  
The first download takes ~30–60s. Subsequent runs load from cache instantly.

In [26]:
def format_laptime(td):
    """Format a timedelta as M:SS.mmm, matching F1 timing display."""
    total_seconds = td.total_seconds()
    minutes = int(total_seconds // 60)
    seconds = total_seconds % 60
    return f'{minutes}:{seconds:06.3f}'

def get_fastest_lap(session, driver_code):
    """Return the single fastest valid lap for a driver in this session."""
    lap = session.laps.pick_driver(driver_code).pick_fastest()
    if lap is None or lap.empty:
        raise ValueError(f'No valid lap found for {driver_code}')
    return lap


print(f'Loading {YEAR} {GRAND_PRIX} GP — {SESSION_TYPE} ...')

session = fastf1.get_session(YEAR, GRAND_PRIX, SESSION_TYPE)
session.load(laps=True, telemetry=True, weather=False, messages=False)

lap_d1 = get_fastest_lap(session, DRIVER_1)
lap_d2 = get_fastest_lap(session, DRIVER_2)

t1 = lap_d1['LapTime']
t2 = lap_d2['LapTime']
delta_total = (t1 - t2).total_seconds()
faster = DRIVER_1 if delta_total < 0 else DRIVER_2
sign = '+' if delta_total > 0 else ''

print(f'Session: {session.event["EventName"]} — {session.name}')
print(f'{DRIVER_1}: {format_laptime(t1)}')
print(f'{DRIVER_2}: {format_laptime(t2)}')
print(f'Gap: {sign}{delta_total:.3f}s ({faster} faster)')

Loading 2026 China GP — Q ...


core           INFO 	Loading data for Chinese Grand Prix - Qualifying [v3.8.1]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for _extended_timing_data. Loading data...
_api           INFO 	Fetching timing data...
_api           INFO 	Parsing timing data...
req            INFO 

Session: Chinese Grand Prix — Qualifying
RUS: 1:32.286
ANT: 1:32.064
Gap: +0.222s (ANT faster)


## Cell 4 — Extract & Align Telemetry

Telemetry is sampled at ~18 Hz (one point every ~0.055s).  
We resample both drivers to a shared distance axis so every point is comparable.

In [27]:
tel_d1 = lap_d1.get_telemetry()
tel_d2 = lap_d2.get_telemetry()

# Resample both drivers onto a shared distance axis so every index
# represents the same physical point on track for both cars
RESOLUTION = 500
min_dist = max(tel_d1['Distance'].min(), tel_d2['Distance'].min())
max_dist = min(tel_d1['Distance'].max(), tel_d2['Distance'].max())
common_dist = np.linspace(min_dist, max_dist, RESOLUTION)

def resample_channel(telemetry_df, channel, new_dist):
    """Interpolate a telemetry channel onto a new distance axis."""
    return np.interp(new_dist, telemetry_df['Distance'].values, telemetry_df[channel].values)

CHANNELS = ['Speed', 'Throttle', 'Brake', 'nGear', 'RPM']

aligned = {'Distance': common_dist}
for ch in CHANNELS:
    if ch in tel_d1.columns and ch in tel_d2.columns:
        aligned[f'{ch}_D1'] = resample_channel(tel_d1, ch, common_dist)
        aligned[f'{ch}_D2'] = resample_channel(tel_d2, ch, common_dist)

df = pd.DataFrame(aligned)

# Cumulative time delta: derived from physics — time = distance / speed
# dt = (1/v1 - 1/v2) * dd, integrated across the lap
# Negative = Driver 1 ahead, Positive = Driver 2 ahead
v1 = np.where(df['Speed_D1'].values / 3.6 < 1, 1, df['Speed_D1'].values / 3.6)
v2 = np.where(df['Speed_D2'].values / 3.6 < 1, 1, df['Speed_D2'].values / 3.6)
dd = np.diff(common_dist, prepend=common_dist[0])
df['Delta'] = np.cumsum((1/v1 - 1/v2) * dd)

# Approximate sector boundaries at lap distance thirds
total_dist = common_dist[-1]
sector_bounds = [total_dist / 3, 2 * total_dist / 3]

print(f'Telemetry aligned — {RESOLUTION} points over {total_dist:.0f}m')
print(f'Channels: {[c for c in CHANNELS if f"{c}_D1" in df.columns]}')

Telemetry aligned — 500 points over 5408m
Channels: ['Speed', 'Throttle', 'Brake', 'nGear', 'RPM']


## Cell 5 — Visualizations

Four stacked interactive charts with a shared X axis (track distance).  
Hover to inspect exact values at any point on track. Zoom and pan freely.

In [28]:
fig = make_subplots(
    rows=4, cols=1,
    shared_xaxes=True,
    row_heights=[0.38, 0.22, 0.20, 0.20],
    vertical_spacing=0.04,
    subplot_titles=[
        'Speed (km/h)',
        f'Delta Time (s)  —  negative = {DRIVER_1} ahead',
        'Throttle (%)',
        'Brake Pressure (%)',
    ]
)

dist = df['Distance'].values

def add_sector_shading(fig, sector_bounds, total_dist, n_rows=4):
    """Add alternating sector shading and dashed boundary lines."""
    fill_colors = ['rgba(255,255,255,0.03)', 'rgba(255,255,255,0.0)', 'rgba(255,255,255,0.03)']
    starts = [0] + sector_bounds
    ends   = sector_bounds + [total_dist]
    for row in range(1, n_rows + 1):
        for i, (s, e) in enumerate(zip(starts, ends)):
            fig.add_vrect(x0=s, x1=e, fillcolor=fill_colors[i], line_width=0, row=row, col=1)
        for b in sector_bounds:
            fig.add_vline(x=b, line_dash='dot', line_color='rgba(255,255,255,0.15)', line_width=1, row=row, col=1)
    return fig


# Row 1 — Speed traces with fill between curves
fig.add_trace(go.Scatter(
    x=dist, y=df['Speed_D1'], name=DRIVER_1,
    line=dict(color=COLOR_D1, width=1.8), hovertemplate='%{y:.0f} km/h'
), row=1, col=1)
fig.add_trace(go.Scatter(
    x=dist, y=df['Speed_D2'], name=DRIVER_2,
    line=dict(color=COLOR_D2, width=1.8), hovertemplate='%{y:.0f} km/h'
), row=1, col=1)
fig.add_trace(go.Scatter(
    x=dist, y=df['Speed_D1'],
    fill=None, mode='lines', line=dict(width=0), showlegend=False, hoverinfo='skip'
), row=1, col=1)
fig.add_trace(go.Scatter(
    x=dist, y=df['Speed_D2'],
    fill='tonexty', fillcolor='rgba(0,210,255,0.07)',
    mode='lines', line=dict(width=0), showlegend=False, hoverinfo='skip'
), row=1, col=1)

# Row 2 — Cumulative delta, coloured by which driver is ahead
delta_d1_ahead = np.where(df['Delta'] < 0,  df['Delta'], np.nan)
delta_d2_ahead = np.where(df['Delta'] >= 0, df['Delta'], np.nan)

fig.add_trace(go.Scatter(
    x=dist, y=delta_d1_ahead, name=f'{DRIVER_1} ahead',
    line=dict(color=COLOR_D1, width=2),
    fill='tozeroy', fillcolor='rgba(0,210,255,0.12)',
    hovertemplate='%{y:.3f}s'
), row=2, col=1)
fig.add_trace(go.Scatter(
    x=dist, y=delta_d2_ahead, name=f'{DRIVER_2} ahead',
    line=dict(color=COLOR_D2, width=2),
    fill='tozeroy', fillcolor='rgba(255,107,53,0.12)',
    hovertemplate='%{y:.3f}s'
), row=2, col=1)
fig.add_hline(y=0, line_dash='solid', line_color='rgba(255,255,255,0.2)', line_width=0.8, row=2, col=1)

# Row 3 — Throttle
fig.add_trace(go.Scatter(
    x=dist, y=df['Throttle_D1'],
    line=dict(color=COLOR_D1, width=1.4), showlegend=False, hovertemplate='%{y:.0f}%'
), row=3, col=1)
fig.add_trace(go.Scatter(
    x=dist, y=df['Throttle_D2'],
    line=dict(color=COLOR_D2, width=1.4), showlegend=False, hovertemplate='%{y:.0f}%'
), row=3, col=1)

# Row 4 — Brake
fig.add_trace(go.Scatter(
    x=dist, y=df['Brake_D1'].astype(float) * 100,
    line=dict(color=COLOR_D1, width=1.4), showlegend=False, hovertemplate='%{y:.0f}%'
), row=4, col=1)
fig.add_trace(go.Scatter(
    x=dist, y=df['Brake_D2'].astype(float) * 100,
    line=dict(color=COLOR_D2, width=1.4), showlegend=False, hovertemplate='%{y:.0f}%'
), row=4, col=1)

fig = add_sector_shading(fig, sector_bounds, total_dist)

# Sector labels along the top of the speed chart
sector_midpoints = [
    (0 + sector_bounds[0]) / 2,
    (sector_bounds[0] + sector_bounds[1]) / 2,
    (sector_bounds[1] + total_dist) / 2
]
for i, mid in enumerate(sector_midpoints):
    fig.add_annotation(
        x=mid, y=1.06, text=f'S{i+1}',
        xref='x1', yref='paper',
        showarrow=False,
        font=dict(color='rgba(255,255,255,0.4)', size=10)
    )

fig.update_layout(
    template=CHART_THEME,
    title=dict(
        text=(
            f'<b>{YEAR} {GRAND_PRIX} GP — {SESSION_TYPE}</b>   '
            f'<span style="color:{COLOR_D1}">{DRIVER_1} {format_laptime(t1)}</span> '
            f'vs '
            f'<span style="color:{COLOR_D2}">{DRIVER_2} {format_laptime(t2)}</span> '
            f'({sign}{delta_total:.3f}s)'
        ),
        font=dict(size=14)
    ),
    height=750,
    margin=dict(l=60, r=30, t=70, b=50),
    hovermode='x unified',
    legend=dict(orientation='h', x=0, y=1.04, font=dict(size=11)),
    plot_bgcolor='rgba(0,0,0,0)',
    paper_bgcolor='rgba(0,0,0,0)'
)
fig.update_xaxes(title_text='Distance (m)', row=4, col=1)

fig.show()

if SAVE_HTML:
    fname = EXPORT_DIR / f'{YEAR}_{GRAND_PRIX}_{SESSION_TYPE}_{DRIVER_1}_vs_{DRIVER_2}.html'
    fig.write_html(str(fname))
    print(f'Saved: {fname.resolve()}')

Saved: C:\Users\Lenovo User\Desktop\f1-telemetry\notebooks\exports\2026_China_Q_RUS_vs_ANT.html


## Cell 6 — Analytical Insights

Automated text summary comparing the two laps.

In [29]:
def sector_delta(df, start, end):
    """Return net time delta gained/lost within a sector distance window."""
    mask = (df['Distance'] >= start) & (df['Distance'] < end)
    s = df[mask]
    return s['Delta'].iloc[-1] - s['Delta'].iloc[0]


starts = [0,                sector_bounds[0], sector_bounds[1]]
ends   = [sector_bounds[0], sector_bounds[1], total_dist]
sector_deltas = [sector_delta(df, s, e) for s, e in zip(starts, ends)]

max_speed_d1    = df['Speed_D1'].max()
max_speed_d2    = df['Speed_D2'].max()
avg_speed_d1    = df['Speed_D1'].mean()
avg_speed_d2    = df['Speed_D2'].mean()
avg_throttle_d1 = df['Throttle_D1'].mean()
avg_throttle_d2 = df['Throttle_D2'].mean()
brake_pct_d1    = (df['Brake_D1'].astype(float) > 0.1).mean() * 100
brake_pct_d2    = (df['Brake_D2'].astype(float) > 0.1).mean() * 100

SEP = '-' * 52

print(f'TELEMETRY REPORT — {YEAR} {GRAND_PRIX} GP ({SESSION_TYPE})')
print(SEP)

print(f'\nLAP TIMES')
print(f'  {DRIVER_1}: {format_laptime(t1)}')
print(f'  {DRIVER_2}: {format_laptime(t2)}')
print(f'  {faster} faster by {abs(delta_total):.3f}s')

print(f'\nSECTOR BREAKDOWN')
for i, d in enumerate(sector_deltas):
    sector_winner = DRIVER_1 if d < 0 else DRIVER_2
    print(f'  Sector {i+1}: {sector_winner} +{abs(d):.3f}s')

print(f'\nSPEED')
print(f'  Top speed   — {DRIVER_1}: {max_speed_d1:.0f} km/h  |  {DRIVER_2}: {max_speed_d2:.0f} km/h')
print(f'  Avg speed   — {DRIVER_1}: {avg_speed_d1:.0f} km/h  |  {DRIVER_2}: {avg_speed_d2:.0f} km/h')
top_speed_winner = DRIVER_1 if max_speed_d1 > max_speed_d2 else DRIVER_2
print(f'  {top_speed_winner} carries more peak speed (+{abs(max_speed_d1 - max_speed_d2):.1f} km/h)')

print(f'\nTHROTTLE')
print(f'  Avg throttle — {DRIVER_1}: {avg_throttle_d1:.0f}%  |  {DRIVER_2}: {avg_throttle_d2:.0f}%')
throttle_winner = DRIVER_1 if avg_throttle_d1 > avg_throttle_d2 else DRIVER_2
print(f'  {throttle_winner} applies more average throttle across the lap')

print(f'\nBRAKING')
print(f'  Brake zone coverage — {DRIVER_1}: {brake_pct_d1:.0f}%  |  {DRIVER_2}: {brake_pct_d2:.0f}%')
brake_winner = DRIVER_1 if brake_pct_d1 < brake_pct_d2 else DRIVER_2
print(f'  {brake_winner} spends less of the lap on the brakes')

sector_wins_d1 = sum(1 for d in sector_deltas if d < 0)
sector_wins_d2 = 3 - sector_wins_d1

print(f'\nSUMMARY')
print(f'  {faster} took the overall advantage by {abs(delta_total):.3f}s.')
print(f'  Sector wins: {DRIVER_1} {sector_wins_d1}/3  |  {DRIVER_2} {sector_wins_d2}/3')
print(f'  {throttle_winner} showed more aggressive throttle application out of corners.')
print(f'  {brake_winner} used a sharper braking style with fewer total braking metres.')
print(f'\n{SEP}')

TELEMETRY REPORT — 2026 China GP (Q)
----------------------------------------------------

LAP TIMES
  RUS: 1:32.286
  ANT: 1:32.064
  ANT faster by 0.222s

SECTOR BREAKDOWN
  Sector 1: ANT +0.185s
  Sector 2: RUS +0.229s
  Sector 3: RUS +0.275s

SPEED
  Top speed   — RUS: 333 km/h  |  ANT: 332 km/h
  Avg speed   — RUS: 239 km/h  |  ANT: 239 km/h
  RUS carries more peak speed (+1.0 km/h)

THROTTLE
  Avg throttle — RUS: 79%  |  ANT: 78%
  RUS applies more average throttle across the lap

BRAKING
  Brake zone coverage — RUS: 14%  |  ANT: 19%
  RUS spends less of the lap on the brakes

SUMMARY
  ANT took the overall advantage by 0.222s.
  Sector wins: RUS 2/3  |  ANT 1/3
  RUS showed more aggressive throttle application out of corners.
  RUS used a sharper braking style with fewer total braking metres.

----------------------------------------------------


## Cell 7 — Gear Map & RPM Comparison

In [30]:
fig2 = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    row_heights=[0.5, 0.5],
    vertical_spacing=0.06,
    subplot_titles=['Gear Selection', 'RPM']
)

# Step-line shape ('hv') reflects instantaneous gear shifts accurately
fig2.add_trace(go.Scatter(
    x=dist, y=df['nGear_D1'], name=DRIVER_1,
    line=dict(color=COLOR_D1, width=1.6, shape='hv'), hovertemplate='Gear %{y}'
), row=1, col=1)
fig2.add_trace(go.Scatter(
    x=dist, y=df['nGear_D2'], name=DRIVER_2,
    line=dict(color=COLOR_D2, width=1.6, shape='hv'), hovertemplate='Gear %{y}'
), row=1, col=1)

fig2.add_trace(go.Scatter(
    x=dist, y=df['RPM_D1'],
    line=dict(color=COLOR_D1, width=1.4), showlegend=False, hovertemplate='%{y:.0f} RPM'
), row=2, col=1)
fig2.add_trace(go.Scatter(
    x=dist, y=df['RPM_D2'],
    line=dict(color=COLOR_D2, width=1.4), showlegend=False, hovertemplate='%{y:.0f} RPM'
), row=2, col=1)

fig2 = add_sector_shading(fig2, sector_bounds, total_dist, n_rows=2)

fig2.update_layout(
    template=CHART_THEME,
    title=f'{YEAR} {GRAND_PRIX} GP — {SESSION_TYPE} | Gear & RPM | {DRIVER_1} vs {DRIVER_2}',
    height=450,
    hovermode='x unified',
    margin=dict(l=60, r=30, t=60, b=50),
    legend=dict(orientation='h', x=0, y=1.06, font=dict(size=11)),
    plot_bgcolor='rgba(0,0,0,0)',
    paper_bgcolor='rgba(0,0,0,0)'
)
fig2.update_yaxes(title_text='Gear', row=1, tickvals=list(range(1, 9)))
fig2.update_yaxes(title_text='RPM', row=2)
fig2.update_xaxes(title_text='Distance (m)', row=2)

fig2.show()

if SAVE_HTML:
    fname2 = EXPORT_DIR / f'{YEAR}_{GRAND_PRIX}_{SESSION_TYPE}_{DRIVER_1}_vs_{DRIVER_2}_gear_rpm.html'
    fig2.write_html(str(fname2))
    print(f'Saved: {fname2.resolve()}')

Saved: C:\Users\Lenovo User\Desktop\f1-telemetry\notebooks\exports\2026_China_Q_RUS_vs_ANT_gear_rpm.html
